# Traductor Inglés → Español
### Transformer implementado desde cero en PyTorch — Inferencia

Este notebook carga el modelo ya entrenado (pesos `.pth`) y los vocabularios,
y permite traducir cualquier oración en inglés al español.

**No requiere GPU** para traducir oraciones individuales (es rápido también en CPU).

In [16]:
# =========================
# 1. SETUP: MONTAR DRIVE Y CARGAR VOCABULARIOS
# =========================
import os, json, math, re
import torch
import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive

drive.mount("/content/drive")

# --- Rutas del modelo entrenado con OpenSubtitles (1M) ---
PROJ = "/content/drive/MyDrive/Proyecto_traductor"
VOCAB_DIR = os.path.join(PROJ, "vocabularios_opensubtitles_1m")
MODEL_DIR = os.path.join(PROJ, "modelos_opensubtitles_1m")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)

with open(os.path.join(VOCAB_DIR, "vocab_english.json"), encoding="utf-8") as f:
    stoi_en = json.load(f)
with open(os.path.join(VOCAB_DIR, "vocab_spanish.json"), encoding="utf-8") as f:
    stoi_es = json.load(f)
with open(os.path.join(MODEL_DIR, "model_config.json"), encoding="utf-8") as f:
    model_cfg = json.load(f)

itos_es = {v: k for k, v in stoi_es.items()}
PAD_IDX, UNK_IDX, SOS_IDX, EOS_IDX = 0, 1, 2, 3

print(f"Vocabulario inglés : {len(stoi_en):,} palabras")
print(f"Vocabulario español: {len(stoi_es):,} palabras")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dispositivo: cpu
Vocabulario inglés : 28,160 palabras
Vocabulario español: 37,161 palabras


In [17]:
# =========================
# 2. ARQUITECTURA DEL TRANSFORMER (implementada desde cero)
# =========================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term); pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.d_k = d_model // n_heads; self.n_heads = n_heads
        self.w_q = nn.Linear(d_model, d_model); self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model); self.w_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, q, k, v, mask=None):
        B = q.size(0)
        Q = self.w_q(q).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.w_k(k).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.w_v(v).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None: scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = self.dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, -1, self.n_heads * self.d_k)
        return self.w_o(out)

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model))
    def forward(self, x): return self.net(x)

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model); self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))
        return self.norm2(x + self.dropout(self.ff(x)))

class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, dropout=0.1, max_len=100):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout); self.d_model = d_model
    def forward(self, src, mask):
        x = self.embed(src) * math.sqrt(self.d_model)
        x = self.dropout(self.pos_enc(x))
        for layer in self.layers: x = layer(x, mask)
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model); self.norm2 = nn.LayerNorm(d_model); self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, enc_out, src_mask, tgt_mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
        x = self.norm2(x + self.dropout(self.cross_attn(x, enc_out, enc_out, src_mask)))
        return self.norm3(x + self.dropout(self.ff(x)))

class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, dropout=0.1, max_len=100):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout); self.d_model = d_model
    def forward(self, tgt, enc_out, src_mask, tgt_mask):
        x = self.embed(tgt) * math.sqrt(self.d_model)
        x = self.dropout(self.pos_enc(x))
        for layer in self.layers: x = layer(x, enc_out, src_mask, tgt_mask)
        return self.fc_out(x)

class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=256, n_heads=8, d_ff=512,
                 n_layers=3, dropout=0.1, max_len=100, pad_idx=0):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, d_model, n_heads, d_ff, n_layers, dropout, max_len)
        self.decoder = Decoder(tgt_vocab_size, d_model, n_heads, d_ff, n_layers, dropout, max_len)
        self.pad_idx = pad_idx
    def make_src_mask(self, src): return (src != self.pad_idx).unsqueeze(1).unsqueeze(2)
    def make_tgt_mask(self, tgt):
        pad_mask = (tgt != self.pad_idx).unsqueeze(1).unsqueeze(2)
        L = tgt.size(1)
        sub_mask = torch.tril(torch.ones((L, L), device=tgt.device)).bool()
        return pad_mask & sub_mask
    def forward(self, src, tgt):
        src_mask = self.make_src_mask(src); tgt_mask = self.make_tgt_mask(tgt)
        enc_out = self.encoder(src, src_mask)
        return self.decoder(tgt, enc_out, src_mask, tgt_mask)

In [18]:
# =========================
# 3. CARGAR EL MODELO ENTRENADO
# =========================
model = Transformer(
    src_vocab_size=len(stoi_en), tgt_vocab_size=len(stoi_es), **model_cfg, pad_idx=PAD_IDX
).to(DEVICE)

CKPT_PATH = os.path.join(MODEL_DIR, "best_model.pth")
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Modelo cargado correctamente ✅")
print(f"Parámetros entrenables: {n_params:,}")
print(f"Checkpoint: {CKPT_PATH}")

Modelo cargado correctamente ✅
Parámetros entrenables: 31,802,153
Checkpoint: /content/drive/MyDrive/Proyecto_traductor/modelos_opensubtitles_1m/best_model.pth


In [23]:
# =========================
# 4. FUNCIÓN DE TRADUCCIÓN (beam search) + limpieza de salida
# =========================
def limpiar_salida(texto):
    texto = re.sub(r'\s+([.,?!:;])', r'\1', texto)   # quita espacio antes de puntuación
    texto = re.sub(r'¿\s+', '¿', texto)               # pega ¿ a la palabra siguiente
    texto = re.sub(r'¡\s+', '¡', texto)
    return texto[0].upper() + texto[1:] if texto else texto

def traducir(oracion, beam_size=3, max_len=32):
    """Traduce una oración en inglés al español usando el Transformer entrenado."""
    model.eval()
    tokens = tokenize(oracion)[:30]
    src_ids = [SOS_IDX] + [stoi_en.get(t, UNK_IDX) for t in tokens] + [EOS_IDX]
    src_tensor = torch.tensor([src_ids], dtype=torch.long).to(DEVICE)
    src_mask = model.make_src_mask(src_tensor)

    with torch.no_grad():
        enc_out = model.encoder(src_tensor, src_mask)

    beams = [([SOS_IDX], 0.0)]
    for _ in range(max_len):
        candidates = []
        all_done = True
        for ids, log_prob in beams:
            if ids[-1] == EOS_IDX:
                candidates.append((ids, log_prob)); continue
            all_done = False
            tgt_tensor = torch.tensor([ids], dtype=torch.long).to(DEVICE)
            tgt_mask = model.make_tgt_mask(tgt_tensor)
            with torch.no_grad():
                out = model.decoder(tgt_tensor, enc_out, src_mask, tgt_mask)
            log_probs = F.log_softmax(out[0, -1], dim=-1)
            topk_probs, topk_ids = log_probs.topk(beam_size)
            for k in range(beam_size):
                candidates.append((ids + [topk_ids[k].item()], log_prob + topk_probs[k].item()))
        if all_done:
            break
        candidates.sort(key=lambda x: x[1], reverse=True)
        beams = candidates[:beam_size]

    best_ids, _ = beams[0]
    palabras = [itos_es.get(i, "<UNK>") for i in best_ids[1:] if i != EOS_IDX]
    return limpiar_salida(" ".join(palabras))

In [24]:
# =========================
# 5. EJEMPLOS DE TRADUCCIÓN
# =========================
ejemplos = [
    "I love you.",
    "What time is it?",
    "Where is the bathroom?",
    "Thank you very much.",
    "I don't understand.",
    "The weather is beautiful today.",
    "Hello woman.",
]

print("=== Ejemplos de traducción ===\n")
for oracion in ejemplos:
    print(f"EN: {oracion}")
    print(f"ES: {traducir(oracion)}\n")

=== Ejemplos de traducción ===

EN: I love you.
ES: Te quiero.

EN: What time is it?
ES: ¿Qué hora es?

EN: Where is the bathroom?
ES: ¿Dónde está el baño?

EN: Thank you very much.
ES: Muchas gracias.

EN: I don't understand.
ES: No entiendo.

EN: The weather is beautiful today.
ES: El clima es hermoso hoy.

EN: Hello woman.
ES: Hola, mujer.



In [35]:
# =========================
# 6. TRADUCE TU PROPIA ORACIÓN
# =========================
texto = input("Escribe una oración en inglés: ")
print(f"\nEN: {texto}")
print(f"ES: {traducir(texto)}")

Escribe una oración en inglés: only i want only

EN: only i want only
ES: Sólo quiero
